In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [2]:
DATA_PATH = Path("../data/raw/PS_20174392719_1491204439457_log.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


##### Validate the dataset
##### Check required columns

In [4]:
required_columns = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns

]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("ALL required column are present.")

ALL required column are present.


In [5]:
print(df["isFraud"].unique())

[0 1]


In [7]:
if not set(df["isFraud"].unique()).issubset({0, 1}):
    raise ValueError ("Unexpected value found in is fraud.")

print("Target validation passed. ")

Target validation passed. 


In [8]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Duplicate rows:
0


##### Separate target from features

In [10]:
X = df.drop(columns =["isFraud"])
y = df["isFraud"]

In [11]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6362620, 10)
y shape: (6362620,)


##### Decide what to do with the columns
This is an important part of Project.
##### nameOrig
Customer/sender identifier
not one-hot encode millions of unique customer IDs.
Therefore:
nameOrig → DROP for initial model

##### nameDest
Same problem:
nameDest → DROP for initial model
They can become useful later for customer behavioral features, but using them directly as categorical features would create an enormous feature space.

##### type
Keep it.
type → categorical feature
Examples:
PAYMENT
TRANSFER
CASH_OUT
DEBIT
CASH_IN

##### step
Keep it initially.
It represents simulation time and may contain temporal information.
Later,  derive additional temporal features if appropriate.

##### Balance and amount features
Keep:
amount
oldbalanceOrg
newbalanceOrig
oldbalanceDest
newbalanceDest
These contain meaningful transaction information.

##### isFlaggedFraud
Keep it for the initial model.
already observed during EDA that it is extremely sparse, so  evaluate whether it actually contributes useful information.

Therefore  initial feature set is:

In [12]:
features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud"
]

X = df[features]
y = df["isFraud"]

##### Train/test split
This is important because preprocessing must not learn information from the test set.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [14]:
print("Training:", X_train.shape)
print("Testing :", X_test.shape)

print("\nTraining fraud rate:")
print(y_train.mean())

print("\nTesting fraud rate:")
print(y_test.mean())

Training: (5090096, 8)
Testing : (1272524, 8)

Training fraud rate:
0.0012907418642005967

Testing fraud rate:
0.0012911347840983745


Define numerical and categorical features

In [15]:
numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud"
]

categorical_features = [
    "type"
]

##### Build the preprocessing pipeline

In [17]:
numeric_pipeline = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "median")),
        ("scaler", StandardScaler())
    ]
)

In [18]:
categorical_pipeline = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy= "most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

##### Fit ONLY on training data

In [23]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [24]:
print("Original training shape:")
print(X_train.shape)

print("\nProcessed training shape:")
print(X_train_processed.shape)

print("\nOriginal test shape:")
print(X_test.shape)

print("\nProcessed test shape:")
print(X_test_processed.shape)

Original training shape:
(5090096, 8)

Processed training shape:
(5090096, 12)

Original test shape:
(1272524, 8)

Processed test shape:
(1272524, 12)


##### Save the preprocessor
Create the artifact directory 

In [25]:
import joblib

ARTIFACT_DIR = Path("../artifacts/preprocessors")
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [26]:
preprocessor_path = ARTIFACT_DIR / "preprocessor.pkl"

joblib.dump(
    preprocessor,
    preprocessor_path
)

print(f"Saved: {preprocessor_path}")

Saved: ..\artifacts\preprocessors\preprocessor.pkl


##### Save the processed datasets

In [27]:
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [28]:
train_processed = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed
)

test_processed = pd.DataFrame(
    X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else X_test_processed
)

###### What  achieved

###### After this stage:

##### Raw PaySim
      ↓
##### Validation
      ↓
##### Remove identifiers
      ↓
##### Train/Test split
      ↓
##### Numerical preprocessing
      ↓
##### Categorical encoding
      ↓
##### Model-ready features